[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/process_tomography/process_tomography_projective.ipynb)

# Quantum Process Tomography in the Projective Bloch Ball

This example computes the experiment of [the process tomography notebook](process_tomography.ipynb), recovering what a device does to every qubit state from the outcome probabilities of four known inputs, with the states of a qubit as the points of a ball in projective space. Every outcome probability is an affine function of the state, and an affine function on space is a plane: a detector's outcome is a plane, a state is a point, and a probability is the pairing of the two.

A state is the point `centre + bloch_vector`, of weight one, the weight read by the plane at infinity, `mv.w & state`. An effect is a plane, and an outcome's probability is `effect & state`. A channel is a map from points to points: a coherent turn is a rotor about the centre, dephasing a chance of a half turn, and relaxation a chance of decay to the north pole. The reconstruction needs no scalar product: each outcome's dual is the point where the other three outcomes' planes meet, each preparation's dual the plane through the other three preparations.

In the notation of quantum computing the channel reads as its Pauli transfer matrix, the real $4\times4$ matrix $R_{ij}=\tfrac12\operatorname{tr}[\sigma_i\,\mathcal E(\sigma_j)]$ acting on $(1,\mathbf r)$, and the setting as that of the generalized probabilistic theories of [Hardy](https://arxiv.org/abs/quant-ph/0101012) and [Barrett](https://arxiv.org/abs/quant-ph/0508211), where states form a cone and effects its dual.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext
from numga.algebras import PGA3D
from examples.util.animation import show_animation
from examples.quantum.process_tomography import render

np.set_printoptions(precision=4, suppress=True)

ga = PGA3D
mv = NumpyContext(ga).multivector
# A state is a point of the Bloch ball, of weight one; an effect is a plane, which reads states.
Point = ga.gatype.antivector()                                            # intrinsic
Effect = ga.gatype.vector()                                               # extrinsic
Channel = ga.gatype((Point, Point))                                       # Point <- Point

# Three devices, each applying the same fixed turn after its noise: none, dephasing, or relaxation.
labels = ("Rotation", "Rotation + dephasing", "Rotation + relaxation")
angles = np.array([0.075, 0.075, 0.075])                                   # radians per use
phase_flip = np.array([0.0, 0.06, 0.0])
loss = np.array([0.0, 0.0, 0.10])
rotation_plane = (mv.yz + mv.zx + mv.xy) / np.sqrt(3)                     # [] Bivector

# The centre of the ball, the completely mixed state, and its poles.
centre = mv.point([0, 0, 0])                                              # [] Point
north = mv.point([0, 0, 1])                                               # [] Point
south = mv.point([0, 0, -1])                                              # [] Point
# Four calibration directions, arranged as a tetrahedron, and the detector's planes through the centre facing them.
tetrahedron = mv.direction([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]) / np.sqrt(3)       # [preparations] Point
analyzers = mv.vector([[1, 1, 1, 0], [1, -1, -1, 0], [-1, 1, -1, 0], [-1, -1, 1, 0]]) / np.sqrt(3)  # [outcomes] Effect

## 1. Preparing, evolving, measuring

Four pure states at the corners of the tetrahedron go through each device. A detector outcome reads a state by a plane: its offset is read by the plane at infinity, the state's weight, and its slope by a plane through the centre facing the outcome's direction. The four effects add up to the plane at infinity, so the four probabilities add up to the weight, one.

The noise is built from what happens to the spin. Dephasing turns it half way about the field with probability `phase_flip`: the state becomes a mixture of itself and its half turn, `mv.xy >> state`. Relaxation sends it to the north pole with probability `loss`. A mixture of decayed and untouched spins would keep `1 - loss` of the spin across the field; an unobserved decay keeps the amplitude `sqrt(1 - loss)` of it, the most coherence a physical channel allows. That bound is what the projective picture does not supply on its own: [the operator algebra's sandwiches](process_tomography.ipynb) derive it, and here it is stated.

In density-matrix notation the effects read as $F_b=\tfrac14(I+\mathbf n_b\cdot\boldsymbol\sigma)$ and the probabilities as $p(b\mid a)=\operatorname{tr}[F_b\,\mathcal E(\rho_a)]$.

In [ ]:
preparations = centre + tetrahedron                                        # [preparations] Point
# The plane at infinity reads a state's weight; with the analyzers' planes, four effects that add up to it.
effects = (mv.w + analyzers) / 4                                           # [outcomes] Effect

# With probability phase_flip, the state is replaced by its half turn about the field.
dephasing = Point + phase_flip * ((mv.xy >> Point) - Point)                # [channels] Point <- Point

# A decayed spin is at north: the plane at infinity reads the state's weight, and north receives it.
decay = north * (mv.w & Point)                                             # [] Point <- Point
# The part of a state across the field: the part a half turn about the field reverses.
transverse = (Point - (mv.xy >> Point)) / 2                                 # [] Point <- Point
# A fraction loss of the way to north, and the surviving coherence across the field kept at sqrt(1 - loss).
damping = Point + loss * (decay - Point) + (np.sqrt(1 - loss) - (1 - loss)) * transverse   # [channels] Point <- Point

rotations = (rotation_plane * (-angles / 2)).exp()                         # [channels] Rotor
# Calling a map on a map composes them; the rotor then turns every output of the noisy map.
device = rotations >> damping(dephasing)                                   # [channels] Point <- Point
outputs = device[:, None](preparations)                                    # [channels, preparations] Point
probabilities = effects & outputs[..., None]                               # [channels, preparations, outcomes] Scalar

render.draw_projective_experiment(preparations, probabilities, labels);

## 2. Recovering the channel

Each outcome's dual is a state: the point where the other three outcomes' planes meet, which those outcomes never report, scaled so its own outcome reports it once. Summing these points, weighted by the measured probabilities, gives back each output state. Each preparation's dual is an effect: the plane through the other three preparations, scaled to read one on its own preparation, so it reads off how much of that preparation a state holds. An output times the reading of its preparation's dual plane is a dyad, and their sum is the whole channel, a map from points to points that applies to any state and composes with any other channel. The reciprocal frames are meets and joins; no scalar product enters.

In matrix notation the dual planes read as the rows of the inverse of the $4\times4$ matrix whose columns are the preparations' homogeneous coordinates.

In [ ]:
# Each preparation or outcome, with the indices of the other three.
other_idx = np.array([[1, 2, 3], [0, 2, 3], [0, 1, 3], [0, 1, 2]])
# Samples of the sphere of pure states, for drawing the channels' images.
n_latitudes = 12
n_longitudes = 2 * n_latitudes

In [ ]:
# The point where the other three effects' planes meet, scaled so its own effect reads one there.
meets = effects[other_idx[:, 0]] ^ effects[other_idx[:, 1]] ^ effects[other_idx[:, 2]]   # [outcomes] Antivector
measurement_dual = meets / (effects & meets)                               # [outcomes] Point
recovered_outputs = (probabilities * measurement_dual).sum(axis=-1)        # [channels, preparations] Point

# The plane through the other three preparations, scaled to read one on its own preparation.
joins = preparations[other_idx[:, 0]] & preparations[other_idx[:, 1]] & preparations[other_idx[:, 2]]   # [preparations] Vector
preparation_dual = joins / (joins & preparations)                          # [preparations] Effect
# Each recovered output times the reading of its preparation's dual: summed, the learned channel. Only the
# probabilities enter this fit.
learned_channel = (recovered_outputs * (preparation_dual & Point)).sum(axis=-1)  # [channels] Point <- Point

# Pure states fill the surface of the Bloch ball: north, tilted down and turned about the field.
tilts = ((mv.z ^ mv.x) * (-np.linspace(0, np.pi, n_latitudes + 1) / 2)).exp()     # [latitudes + 1] Rotor
turns = (mv.xy * (-np.linspace(0, 2 * np.pi, n_longitudes + 1) / 2)).exp()        # [longitudes + 1] Rotor
surface = turns[None, :] >> (tilts[:, None] >> north)                       # [latitudes + 1, longitudes + 1] Point
images = learned_channel[:, None, None](surface)                            # [channels, latitudes + 1, longitudes + 1] Point
probe_images = learned_channel[:, None](preparations)                       # [channels, preparations] Point
render.draw_projective_channels(surface, images, preparations, probe_images, labels);

## 3. Predicting unseen inputs

The learned channel predicts the detector's outcomes for any input, not only the four it was recovered from. The test states lie along the axes, at several distances from the centre, down to the completely mixed state at the centre itself. Dephasing leaves the centre where it is; relaxation moves it towards north, a displacement the channel's reading of the weight, `north * (mv.w & Point)`, carries.

In Bloch-vector notation the channel reads as the affine map $\mathbf r'=A\mathbf r+\mathbf t$.

In [ ]:
# Six held-out directions along the axes, and the distances of their states from the centre: one is pure, zero the
# completely mixed state.
validation_directions = mv.direction([[1, 0, 0], [0, 1, 0], [0, 0, 1],
                                      [-1, 0, 0], [0, -1, 0], [0, 0, -1]])  # [heldout] Point
bloch_lengths = np.array([1.0, 0.7, 0.3, 1.0, 0.7, 0.0])

In [ ]:
heldout_states = centre + validation_directions * bloch_lengths             # [heldout] Point
actual_outputs = device[:, None](heldout_states)                            # [channels, heldout] Point
predicted_outputs = learned_channel[:, None](heldout_states)                # [channels, heldout] Point

# Compare measurable predictions on each held-out state, using the same detector.
observed_probabilities = effects & actual_outputs[..., None]               # [channels, heldout, outcomes] Scalar
predicted_probabilities = effects & predicted_outputs[..., None]           # [channels, heldout, outcomes] Scalar
render.draw_predictions(observed_probabilities, predicted_probabilities, labels);

## 4. Four probes against two

A channel is pinned down by preparations that span the ball: four points in general position join to a volume, and their dual planes exist. Two opposite preparations join only to the line through them, the axis of the field: a device that turned states about that line, or squeezed them across it, would give the same answers on both, and no plane through one of them can single out the other.

In [ ]:
tetrahedral_span = preparations[0] & preparations[1] & preparations[2] & preparations[3]   # [] Scalar
opposite_span = north & south                                              # [] Line
print("Four tetrahedral probes join to a volume:\n", tetrahedral_span.coefficient_table())
print("Two opposite probes join to a line:\n", opposite_span.coefficient_table())

## 5. Predicting repeated uses

Composing the learned channel with itself gives the map for any number of uses, without measuring again. A coherent turn keeps pure states on the sphere; noise contracts it, and relaxation also moves it. The hollow points mark the four preparations, and the filled points their predicted outputs through repeated uses of each device.

In the notation of quantum computing the sequence reads as the power $R^n$ of the Pauli transfer matrix.

In [ ]:
# The number of uses, and the duration of an animation frame.
n_uses = 48
frame_duration = 80

In [ ]:
def powers(channel: Channel, n_uses: int) -> Iterator[Channel]:
    """One application, then two, three, and so on, as composed maps."""
    channel_power = channel                                              # [channels] Point <- Point
    for _ in range(n_uses):
        yield channel_power
        channel_power = channel(channel_power)                           # [channels] Point <- Point


# The input geometry stays fixed. Each frame contains only its two changing images.
frames = ((channel_power[:, None, None](surface), channel_power[:, None](preparations))
          for channel_power in powers(learned_channel, n_uses))
show_animation(render.animate_projective(surface, preparations, frames, labels), "process_tomography_projective",
               frame_duration)

The reconstruction follows [Chuang and Nielsen, *Prescription for experimental determination of the dynamics of a quantum black box*](https://arxiv.org/abs/quant-ph/9610001); the tetrahedral detector is a qubit example of the [symmetric informationally complete measurements of Renes et al.](https://arxiv.org/abs/quant-ph/0310075).